# SymptoSense DDXPlus: symptom triage modeling

This notebook documents the reproducible classical-ML workflow. DDXPlus contains synthetic cases; results do not establish clinical performance. Run `python train_pipeline.py --train-rows 150000` from `ml-service` to rebuild the models and report. The official test split is only used after training/model selection.

## Problem and data source

Predict the dataset pathology from patient evidence plus age and sex, and model the dataset-defined severity proxy. This is not a medical diagnosis. DDXPlus is synthetic and released under CC BY 4.0. INITIAL_EVIDENCE is excluded because the app does not collect it.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from scipy import sparse
sys.path.insert(0, str(Path.cwd()))
from ml_pipeline.ddxplus_data import DATA, load_metadata, load_splits, vocabulary, feature_schema, encode_rows
evidence_meta, conditions = load_metadata()
print('Evidence entries:', len(evidence_meta), 'pathologies:', len(conditions))
print(pd.read_csv(DATA / 'release_train_patients', nrows=5).columns.tolist())

Evidence entries: 223 pathologies: 49
['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']


## EDA and preprocessing

The CSV `EVIDENCES` field is parsed as a list; valued items use `_@_`. Binary and valued evidence are expanded using the evidence metadata schema. Symptom and antecedent features retain distinct prefixes. Sampling is seeded and stratified by PATHOLOGY. Sex is masked to unknown for 20% of training rows. Age bins match the app, assigning age 50 to elderly to resolve its overlapping range.

In [2]:
frames, sample_sizes, split_overlap, source_paths = load_splits(150000)
print(sample_sizes)
print('Exact-record fingerprint overlaps:', split_overlap)
print('Training duplicate evidence patterns:', int(frames['train']['EVIDENCES'].duplicated().sum()))
print(frames['train'].head(10)[['AGE','SEX','PATHOLOGY','EVIDENCES']].to_string(index=False))
columns = feature_schema()
X_train, y_train, _ = encode_rows(frames['train'].head(1000), columns, mask_sex=True)
print('Sparse smoke matrix:', X_train.shape, 'nonzero:', X_train.nnz)

{'train': {'rows_before_sampling': 1025602, 'rows_after_sampling': 150000, 'class_distribution': {'Acute COPD exacerbation / infection': 2584, 'Acute dystonic reactions': 3799, 'Acute laryngitis': 3528, 'Acute otitis media': 3789, 'Acute pulmonary edema': 2782, 'Acute rhinosinusitis': 1987, 'Allergic sinusitis': 3831, 'Anaphylaxis': 4053, 'Anemia': 7404, 'Atrial fibrillation': 3077, 'Boerhaave': 2207, 'Bronchiectasis': 2749, 'Bronchiolitis': 42, 'Bronchitis': 3860, 'Bronchospasm / acute asthma exacerbation': 2907, 'Chagas': 1356, 'Chronic rhinosinusitis': 3010, 'Cluster headache': 3043, 'Croup': 421, 'Ebola': 109, 'Epiglottitis': 2518, 'GERD': 3799, 'Guillain-Barré syndrome': 3344, 'HIV (initial infection)': 4242, 'Influenza': 3920, 'Inguinal hernia': 2960, 'Larygospasm': 1611, 'Localized edema': 4068, 'Myasthenia gravis': 2716, 'Myocarditis': 1621, 'PSVT': 2747, 'Pancreatic neoplasm': 2757, 'Panic attack': 3658, 'Pericarditis': 3332, 'Pneumonia': 3767, 'Possible NSTEMI / STEMI': 3109,

Sparse smoke matrix: (1000, 980) nonzero: 22278


## Model training and tuning

The single training entry point compares Logistic Regression, Multinomial Naive Bayes, Random Forest, and XGBoost with five-fold stratified CV on the training sample. The two CV leaders receive randomized searches using training folds only; the official validation split chooses the final estimator. The test set is untouched until final evaluation. For a complete retrain, run the command above; the next cell reads the generated report.

In [3]:
report_path = Path('../ml-service/reports/ml_report.json') if Path.cwd().name == 'notebooks' else Path('reports/ml_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('CV comparison:', json.dumps(report['training']['comparison'], indent=2))
    print('Tuning:', json.dumps(report['training']['hyperparameters'], indent=2))
else:
    print('No report found yet. Run python train_pipeline.py --train-rows 150000 from ml-service.')

CV comparison: [
  {
    "model": "XGBoost",
    "metrics": {
      "accuracy": {
        "mean": 0.99694,
        "std": 0.0003101254226563514,
        "folds": [
          0.9964,
          0.9972666666666666,
          0.9968333333333333,
          0.997,
          0.9972
        ]
      },
      "precision_macro": {
        "mean": 0.9966449704873093,
        "std": 0.0007759962079177294,
        "folds": [
          0.9964935614933376,
          0.9974042674854302,
          0.9970045531622741,
          0.9971136254304336,
          0.9952088448650698
        ]
      },
      "recall_macro": {
        "mean": 0.9957360626890072,
        "std": 0.0005299485286156744,
        "folds": [
          0.9947630431846353,
          0.9963216767912154,
          0.9957646849628924,
          0.9957583961728296,
          0.9960725123334627
        ]
      },
      "f1_macro": {
        "mean": 0.9959909221897554,
        "std": 0.0005214505429484554,
        "folds": [
          0.9953725

## Final held-out evaluation, baselines and robustness

The report contains accuracy, macro and weighted precision/recall/F1, top-3/top-5, one-vs-rest macro ROC-AUC, confusion matrix, per-class metrics, train/test gap, majority and nearest-neighbour baselines, calibration comparison, evidence-drop and wrong-evidence tests, and a noise-augmented model. These are dataset performance measures only.

In [4]:
if 'report' in globals():
    print('Test metrics:', json.dumps(report['evaluation']['metrics'], indent=2))
    print('Top-k:', report['evaluation']['topk'])
    print('Baselines:', json.dumps(report['baselines'], indent=2))
    print('Noise robustness:', json.dumps(report['noise_robustness'], indent=2))
    print('Calibration:', json.dumps(report['calibration'], indent=2))

Test metrics: {
  "accuracy": 0.9972047164057736,
  "precision_macro": 0.9972715167373772,
  "recall_macro": 0.995710261382543,
  "f1_macro": 0.9963017535132446,
  "precision_weighted": 0.9974147800121197,
  "recall_weighted": 0.9972047164057736,
  "f1_weighted": 0.997167347899425,
  "top_3_accuracy": 1.0,
  "top_5_accuracy": 1.0,
  "macro_auc_ovr": 0.9999734586593859,
  "train_accuracy": 0.9979133333333333,
  "train_test_accuracy_gap": 0.0007086169275597465
}
Top-k: {'top_3': 1.0, 'top_5': 1.0}
Baselines: {
  "majority_class": {
    "accuracy": 0.0624,
    "test_rows": 5000,
    "class": "URTI"
  },
  "nearest_neighbour": {
    "accuracy": 0.9778,
    "top3_accuracy": 0.9958,
    "train_rows": 30000,
    "test_rows": 5000,
    "neighbors": 3,
    "distance": "cosine"
  },
  "interpretation": "Majority-class score is a naive floor; nearest-neighbour tests whether local evidence-pattern similarity beats the selected classical models. Nearest-neighbour evaluation uses a deterministic bou

## Explainability and urgency limitations

Global feature attribution and local top-five contributions describe model associations, not causes. Urgency labels use DDXPlus pathology severity, whose scale direction is lower-number-is-more-severe; the classifier is not clinician validated. Red-flag overrides and the disease safety floors are project safety choices, not clinical guidelines. The rule engine remains the triage authority.

In [5]:
if 'report' in globals():
    print('Top global features:', json.dumps(report['explainability']['global_importance'], indent=2))
    print('Severity mapping:', json.dumps(report['severity_table'], indent=2))
    print('Limitations:', json.dumps(report['limitations'], indent=2))

Top global features: [
  {
    "symptom": "symptom:E_53",
    "importance": 0.2774485647678375
  },
  {
    "symptom": "symptom:E_57:@:V_123",
    "importance": 0.16193166375160217
  },
  {
    "symptom": "symptom:E_151",
    "importance": 0.13128727674484253
  },
  {
    "symptom": "symptom:E_131:@:V_10",
    "importance": 0.11811583489179611
  },
  {
    "symptom": "symptom:E_130:@:V_138",
    "importance": 0.08134754002094269
  },
  {
    "symptom": "symptom:E_130:@:V_156",
    "importance": 0.07417554408311844
  },
  {
    "symptom": "antecedent:E_196",
    "importance": 0.06608401238918304
  },
  {
    "symptom": "symptom:E_201",
    "importance": 0.06304381042718887
  },
  {
    "symptom": "antecedent:E_120",
    "importance": 0.059861697256565094
  },
  {
    "symptom": "symptom:E_194",
    "importance": 0.0570739321410656
  },
  {
    "symptom": "antecedent:E_146",
    "importance": 0.05110177770256996
  },
  {
    "symptom": "symptom:E_155",
    "importance": 0.049246925860643